# 01 - Transformar JSON (SERCOP / OCDS) a Parquet

**Objetivo de este notebook:** convertir los archivos JSON crudos de SERCOP (formato OCDS) que están en
`00_json_crudo/` a archivos Parquet limpios y consolidados en `01_parquet/`, dejando documentado en el
propio notebook:

- La estructura real de los datos (no asumida).
- Las decisiones de limpieza tomadas y su justificación.
- Los valores nulos encontrados y por qué ocurren.
- El RUC del proveedor y de la entidad compradora como llave primaria para cruces posteriores
  (con el catastro de RUC del SRI y con el listado de "empresas fantasmas").

**Regla de trabajo:** este notebook se ejecuta **por pasos**. Cada paso imprime información de
diagnóstico (`DESCRIBE`, conteos, `head`, valores únicos, etc.). Antes de avanzar al siguiente paso,
revisa la salida impresa — si algo no coincide con lo que se documenta aquí (nombres de campos, tipos,
formato de fechas, etc.), se ajusta el código de ese paso antes de continuar. No se asume estructura
que no se haya confirmado con datos reales.


## Paso 0 — Configuración e imports

Ajusta `CARPETA_JSON` y `CARPETA_PARQUET` si tu estructura de carpetas es distinta a la de
`SERCOP_FINAL/`. El resto del notebook no debería necesitar más cambios de rutas.


In [ ]:
import duckdb
import glob
import os
from pathlib import Path

# --- Rutas (corregidas según tu estructura real en C:\SERCOP_FINAL) ---
CARPETA_JSON = r"C:\SERCOP_FINAL\00_json_crudo"
CARPETA_PARQUET = r"C:\SERCOP_FINAL\01_parquet"

# Crear carpeta de salida si no existe
os.makedirs(CARPETA_PARQUET, exist_ok=True)

# Conexión DuckDB (en memoria, no se guarda una base .db)
con = duckdb.connect(database=':memory:')

print(f"Carpeta JSON: {CARPETA_JSON}")
print(f"¿Existe la carpeta?: {os.path.isdir(CARPETA_JSON)}\n")

# --- Diagnóstico crudo: qué hay realmente en la carpeta (sin filtrar por extensión) ---
if os.path.isdir(CARPETA_JSON):
    contenido_crudo = os.listdir(CARPETA_JSON)
    print(f"Archivos/carpetas encontrados sin filtrar (os.listdir): {len(contenido_crudo)}")
    for nombre in contenido_crudo:
        ruta_completa = os.path.join(CARPETA_JSON, nombre)
        if os.path.isfile(ruta_completa):
            tam_mb = os.path.getsize(ruta_completa) / (1024 * 1024)
            print(f"  - {nombre:40s}  {tam_mb:10.2f} MB")
        else:
            print(f"  - {nombre:40s}  (carpeta)")
else:
    print("OJO: la carpeta no existe con esa ruta. Revisa el path exacto en el explorador de Windows.")

print()

# --- Búsqueda con glob por extensión (.json y .jsonl) ---
archivos_json = sorted(glob.glob(os.path.join(CARPETA_JSON, "*.json")) +
                        glob.glob(os.path.join(CARPETA_JSON, "*.jsonl")))

print(f"Archivos .json/.jsonl encontrados por glob: {len(archivos_json)}")
for f in archivos_json:
    tam_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  - {os.path.basename(f):40s}  {tam_mb:10.2f} MB")

## Paso 1 — Inspeccionar la estructura real de UN archivo

Antes de escribir la función de extracción para todos los años, revisamos la estructura real de un
solo archivo: sus columnas de primer nivel (con `DESCRIBE`), y una fila completa de ejemplo (con
`LIMIT 1`), para ver cómo vienen anidados los campos que nos interesan (fechas, montos, proveedor,
entidad compradora, RUC, tipo de contratación, número de oferentes, etc.).

**IMPORTANTE:** ejecuta esta celda y pega el resultado completo (la lista de columnas y el registro de
ejemplo) para confirmar o ajustar la función de extracción del Paso 2. Los nombres de campo que se usan
más abajo (`releases`, `tender`, `buyer`, `awards`, etc.) siguen el estándar OCDS, pero SERCOP puede
tener variantes o campos adicionales/faltantes que solo se ven con datos reales.


In [ ]:
# --- Lista final de archivos a procesar (excluyendo el duplicado releases_2024.json) ---
archivos_json = [f for f in archivos_json if os.path.basename(f) != "releases_2024.json"]

print(f"Archivos a procesar (tras excluir duplicado 2024): {len(archivos_json)}")
for f in archivos_json:
    print(f"  - {os.path.basename(f)}")

In [ ]:
# --- Paso 1: inspeccionar la estructura real de UN archivo ---

# Tomamos un archivo chico para que la inspección sea rápida (2026 es el más liviano, 913 MB)
ARCHIVO_PRUEBA = [f for f in archivos_json if "2026" in os.path.basename(f)][0]
print(f"Archivo de prueba: {ARCHIVO_PRUEBA}\n")

# --- 1a. Estructura de columnas de primer nivel ---
try:
    describe_result = con.execute(f"""
        DESCRIBE
        SELECT * FROM read_json_auto('{ARCHIVO_PRUEBA}', format='newline_delimited')
    """).df()
    formato_detectado = "newline_delimited"
except Exception as e:
    print("Fallo como newline_delimited, probando como JSON normal (auto)...")
    print(f"  (error: {e})\n")
    describe_result = con.execute(f"""
        DESCRIBE
        SELECT * FROM read_json_auto('{ARCHIVO_PRUEBA}')
    """).df()
    formato_detectado = "auto"

print(f"Formato detectado: {formato_detectado}")
print(f"Columnas de primer nivel encontradas: {len(describe_result)}\n")
print(describe_result.to_string())

In [ ]:
# --- 1b. Un registro completo de ejemplo, para ver los campos anidados ---
import json as _json

if formato_detectado == "newline_delimited":
    fuente = f"read_json_auto('{ARCHIVO_PRUEBA}', format='newline_delimited')"
else:
    fuente = f"read_json_auto('{ARCHIVO_PRUEBA}')"

fila_ejemplo = con.execute(f"SELECT * FROM {fuente} LIMIT 1").fetchdf()

print(f"Número de filas en la muestra: {len(fila_ejemplo)}")
print("\nTipos de dato de primer nivel:")
print(fila_ejemplo.dtypes)

print("\nRegistro de ejemplo completo (como diccionario):")
registro_dict = fila_ejemplo.iloc[0].to_dict()
print(_json.dumps(registro_dict, indent=2, default=str, ensure_ascii=False)[:5000])
print("\n... (truncado a 5000 caracteres si el registro es más largo)")

In [ ]:
# --- 1c. Conteo de filas del archivo de prueba ---
n_filas = con.execute(f"SELECT COUNT(*) AS n FROM {fuente}").fetchone()[0]
print(f"Filas (releases) en {os.path.basename(ARCHIVO_PRUEBA)}: {n_filas:,}")

## Paso 2 — Función de extracción (punto de partida, a ajustar según el Paso 1)

Esta consulta SQL es un **punto de partida** basado en el estándar OCDS. Extrae los campos que
necesitamos para el análisis: identificadores, fechas, entidad compradora, proveedor (con su RUC),
tipo y método de contratación, monto, número de oferentes y duración del proceso.

Se deja cada campo comentado con su ruta de origen para poder ajustarlo fila por fila según lo que haya
mostrado el Paso 1. **No ejecutar en todos los archivos todavía** — primero se prueba solo contra
`ARCHIVO_PRUEBA` para confirmar que cada columna se extrae correctamente.


In [ ]:
def query_extraccion(ruta_archivo):
    """
    Extrae los campos relevantes de un archivo JSON OCDS de SERCOP (array de paquetes,
    cada uno con una lista 'releases'). Usa UNNEST para explotar releases a nivel de fila.
    """
    ruta_norm = ruta_archivo.replace("\\", "/")
    return f"""
        SELECT
            r.ocid                                          AS ocid,
            r.id                                             AS release_id,
            r.date                                           AS fecha_release,
            r.tag                                            AS etapas,

            -- Entidad compradora (RUC crudo, se limpia en el notebook de ETL)
            r.buyer.id                                       AS entidad_id,
            r.buyer.name                                     AS entidad_nombre,

            -- Proveedor adjudicado (primer award, primer supplier -- RUC crudo)
            r.awards[1].suppliers[1].id                      AS proveedor_id,
            r.awards[1].suppliers[1].name                    AS proveedor_nombre,

            -- Tender / proceso de contratación
            r.tender.id                                      AS tender_id,
            r.tender.title                                   AS tender_titulo,
            r.tender.status                                  AS tender_status,
            r.tender.procurementMethod                       AS metodo_contratacion,
            r.tender.procurementMethodDetails                AS tipo_contratacion,
            r.tender.mainProcurementCategory                 AS categoria_principal,
            r.tender.value.amount                            AS monto_tender,
            r.awards[1].value.amount                         AS monto_award,
            r.tender.numberOfTenderers                       AS n_oferentes,
            r.tender.tenderPeriod.startDate                  AS fecha_inicio_tender,
            r.tender.tenderPeriod.endDate                    AS fecha_fin_tender,

            -- Conteos estructurales (para variables de riesgo más adelante)
            len(r.awards)                                    AS n_awards,
            len(r.parties)                                   AS n_parties,
            len(r.contracts)                                 AS n_contracts

        FROM read_json_auto('{ruta_norm}', format='array') AS t,
        UNNEST(t.releases) AS u(r)
    """

# Prueba contra el archivo de 2026 (el más liviano)
prueba_extraccion = con.execute(query_extraccion(ARCHIVO_PRUEBA)).fetchdf()

print(f"Filas extraídas de 2026: {len(prueba_extraccion):,}")
print(f"\nColumnas resultantes: {list(prueba_extraccion.columns)}\n")
prueba_extraccion.head(5)

In [ ]:
# --- Diagnóstico de nulos en la muestra, columna por columna ---
print(f"{'columna':30s} {'% nulos':>10s}")
print("-" * 42)
for col in prueba_extraccion.columns:
    pct_nulo = prueba_extraccion[col].isna().mean() * 100
    print(f"{col:30s} {pct_nulo:9.1f}%")

## Paso 3 — Procesar todos los archivos y consolidar en Parquet

Una vez confirmada la extracción contra el archivo de muestra, se repite para todos los archivos
encontrados (uno por año, típicamente), se guarda un Parquet por año, y luego se consolidan todos en un
único archivo `sercop_consolidado.parquet`.

Se usa `union_by_name=True` al consolidar para tolerar que algún año tenga columnas ligeramente
distintas sin que falle la unión.


In [ ]:
resumen_por_archivo = []

for ruta in archivos_json:
    nombre_base = Path(ruta).stem
    ruta_salida = os.path.join(CARPETA_PARQUET, f"{nombre_base}.parquet")

    print(f"Procesando {os.path.basename(ruta)} ...")

    con.execute(f"""
        COPY ({query_extraccion(ruta, formato_detectado)})
        TO '{ruta_salida}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{ruta_salida}')").fetchone()[0]
    tam_mb = os.path.getsize(ruta_salida) / (1024 * 1024)
    resumen_por_archivo.append({"archivo": nombre_base, "filas": n, "tam_mb": round(tam_mb, 2)})
    print(f"  -> {ruta_salida}  ({n:,} filas, {tam_mb:.2f} MB)")

print("\nResumen por archivo:")
for r in resumen_por_archivo:
    print(f"  {r['archivo']:30s} {r['filas']:>12,} filas   {r['tam_mb']:>8.2f} MB")

total_filas_individuales = sum(r["filas"] for r in resumen_por_archivo)
print(f"\nTotal de filas sumando todos los archivos individuales: {total_filas_individuales:,}")


In [ ]:
# --- Consolidar todos los parquet por año en un único archivo ---
RUTA_CONSOLIDADO = os.path.join(CARPETA_PARQUET, "sercop_consolidado.parquet")

patron_parquets = os.path.join(CARPETA_PARQUET, "*.parquet")

con.execute(f"""
    COPY (
        SELECT * FROM read_parquet('{patron_parquets}', union_by_name=True)
        WHERE filename != '{RUTA_CONSOLIDADO}'
    )
    TO '{RUTA_CONSOLIDADO}' (FORMAT PARQUET, COMPRESSION ZSTD)
""") if False else None

# Nota: DuckDB no permite excluir el propio archivo de salida dentro del glob de entrada en la misma
# consulta de forma segura si ya existiera de una corrida anterior. Por eso consolidamos explícitamente
# a partir de los archivos individuales ya listados en resumen_por_archivo:

rutas_individuales = [os.path.join(CARPETA_PARQUET, f"{r['archivo']}.parquet") for r in resumen_por_archivo]
lista_sql = ", ".join([f"'{r}'" for r in rutas_individuales])

con.execute(f"""
    COPY (
        SELECT * FROM read_parquet([{lista_sql}], union_by_name=True)
    )
    TO '{RUTA_CONSOLIDADO}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

total_consolidado = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_CONSOLIDADO}')").fetchone()[0]
print(f"Archivo consolidado: {RUTA_CONSOLIDADO}")
print(f"Filas en el consolidado: {total_consolidado:,}")

# --- Verificación de reconciliación: el consolidado debe tener las mismas filas que la suma individual ---
assert total_consolidado == total_filas_individuales, (
    f"Descuadre: consolidado tiene {total_consolidado:,} filas pero la suma de archivos individuales "
    f"da {total_filas_individuales:,}. Revisar antes de continuar."
)
print("\nOK: el total consolidado coincide exactamente con la suma de los archivos individuales.")


## Paso 4 — EDA pequeño sobre el consolidado

Una revisión rápida del archivo consolidado: dimensiones, rango de fechas, filas por año, % de nulos
por columna, y las categorías más frecuentes de `tipo_contratacion`. Esto sirve como control de calidad
antes de pasar al siguiente notebook (limpieza / ETL).


In [ ]:
import time

resumen_por_archivo = []

for ruta in archivos_json:
    nombre_base = Path(ruta).stem
    ruta_salida = os.path.join(CARPETA_PARQUET, f"{nombre_base}.parquet").replace("\\", "/")

    print(f"Procesando {os.path.basename(ruta)} ...")
    t0 = time.time()

    con.execute(f"""
        COPY ({query_extraccion(ruta)})
        TO '{ruta_salida}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    t1 = time.time()
    n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{ruta_salida}')").fetchone()[0]
    tam_mb = os.path.getsize(ruta_salida) / (1024 * 1024)
    resumen_por_archivo.append({"archivo": nombre_base, "filas": n, "tam_mb": round(tam_mb, 2)})
    print(f"  -> {n:,} filas, {tam_mb:.2f} MB, {t1 - t0:.1f} segundos\n")

print("Resumen por archivo:")
for r in resumen_por_archivo:
    print(f"  {r['archivo']:30s} {r['filas']:>12,} filas   {r['tam_mb']:>8.2f} MB")

total_filas_individuales = sum(r["filas"] for r in resumen_por_archivo)
print(f"\nTotal de filas sumando todos los archivos individuales: {total_filas_individuales:,}")

In [ ]:
RUTA_CONSOLIDADO = os.path.join(CARPETA_PARQUET, "sercop_consolidado.parquet").replace("\\", "/")

rutas_individuales = [os.path.join(CARPETA_PARQUET, f"{r['archivo']}.parquet").replace("\\", "/") for r in resumen_por_archivo]
lista_sql = ", ".join([f"'{r}'" for r in rutas_individuales])

t0 = time.time()
con.execute(f"""
    COPY (
        SELECT * FROM read_parquet([{lista_sql}], union_by_name=True)
    )
    TO '{RUTA_CONSOLIDADO}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
t1 = time.time()

total_consolidado = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_CONSOLIDADO}')").fetchone()[0]
print(f"Archivo consolidado: {RUTA_CONSOLIDADO}")
print(f"Filas en el consolidado: {total_consolidado:,}  ({t1 - t0:.1f} segundos)")

assert total_consolidado == total_filas_individuales, (
    f"Descuadre: consolidado tiene {total_consolidado:,} filas pero la suma de archivos "
    f"individuales da {total_filas_individuales:,}."
)
print("\nOK: el total consolidado coincide con la suma de los archivos individuales.")

Ahora el Paso 4 — EDA pequeño sobre el consolidado:

In [ ]:
RUTA_CONSOLIDADO = os.path.join(CARPETA_PARQUET, "sercop_consolidado.parquet").replace("\\", "/")

# --- Dimensiones generales ---
dims = con.execute(f"""
    SELECT COUNT(*) AS n_filas, COUNT(DISTINCT ocid) AS n_ocid_unicos
    FROM read_parquet('{RUTA_CONSOLIDADO}')
""").fetchdf()
print("Dimensiones del consolidado:")
print(dims.to_string(index=False))

In [ ]:
# --- Rango de fechas y filas por año ---
rango_fechas = con.execute(f"""
    SELECT MIN(fecha_release) AS fecha_min, MAX(fecha_release) AS fecha_max
    FROM read_parquet('{RUTA_CONSOLIDADO}')
""").fetchdf()
print("Rango de fechas (fecha_release):")
print(rango_fechas.to_string(index=False))

filas_por_anio = con.execute(f"""
    SELECT EXTRACT(YEAR FROM fecha_release) AS anio, COUNT(*) AS n_filas
    FROM read_parquet('{RUTA_CONSOLIDADO}')
    GROUP BY 1
    ORDER BY 1
""").fetchdf()
print("\nFilas por año (según fecha_release, no según el archivo de origen):")
print(filas_por_anio.to_string(index=False))

In [ ]:
# --- % de nulos por columna sobre el consolidado completo ---
columnas = con.execute(f"SELECT * FROM read_parquet('{RUTA_CONSOLIDADO}') LIMIT 0").df().columns.tolist()
exprs = ", ".join([f"ROUND(100.0 * SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS {c}" for c in columnas])
pct_nulos = con.execute(f"SELECT {exprs} FROM read_parquet('{RUTA_CONSOLIDADO}')").fetchdf().T
pct_nulos.columns = ["% nulos"]
print("Porcentaje de nulos por columna:")
print(pct_nulos.to_string())

In [ ]:
# --- Top 15 tipo_contratacion (sin normalizar) ---
top_tipo = con.execute(f"""
    SELECT tipo_contratacion, COUNT(*) AS n
    FROM read_parquet('{RUTA_CONSOLIDADO}')
    GROUP BY 1
    ORDER BY n DESC
    LIMIT 15
""").fetchdf()
print("Top 15 tipo_contratacion:")
print(top_tipo.to_string(index=False))

In [ ]:
# --- Muestra de entidad_id / proveedor_id para revisar el formato real del RUC ---
muestra_ruc = con.execute(f"""
    SELECT DISTINCT entidad_id, proveedor_id
    FROM read_parquet('{RUTA_CONSOLIDADO}')
    WHERE entidad_id IS NOT NULL AND proveedor_id IS NOT NULL
    LIMIT 10
""").fetchdf()
print("Muestra de entidad_id / proveedor_id (para ver el formato del RUC antes de limpiarlo):")
print(muestra_ruc.to_string(index=False))

Todo cuadra y confirma la estructura documentada:

2,785,355 filas, 2,785,355 ocid únicos — cero duplicados, cada proceso es una fila.
Rango de fechas 2015-01-05 a 2026-09-04, con el "derrame" ya conocido hacia meses posteriores del año nominal.
Nulos altos en n_oferentes (83.3%), categoria_principal (83.2%), fecha_inicio_tender/fecha_fin_tender (83.2%/92.9%) — confirmado: son procesos por catálogo electrónico / compra directa, que dominan el top_tipo (los primeros 9 lugares son variantes de "Catálogo electrónico").
tipo_contratacion sin normalizar tiene el mismo patrón de siempre ("... en el convenio ...") que ya sabían resolver con SPLIT_PART.

Lo importante — el formato del RUC:

entidad_id → EC-RUC-{13 dígitos} (sin sufijo).
proveedor_id → EC-RUC-{13 dígitos}-{sufijo numérico interno} (sufijo distinto, no es parte del RUC).

Para extraer el RUC puro de 13 dígitos de ambos campos por igual (ignorando el sufijo cuando exista), usa regexp_extract:

In [ ]:
# --- Extracción del RUC puro (13 dígitos) desde entidad_id y proveedor_id ---
muestra_ruc_limpio = con.execute(f"""
    SELECT
        entidad_id,
        regexp_extract(entidad_id, 'EC-RUC-(\\d{{13}})', 1)   AS entidad_ruc,
        proveedor_id,
        regexp_extract(proveedor_id, 'EC-RUC-(\\d{{13}})', 1) AS proveedor_ruc
    FROM read_parquet('{RUTA_CONSOLIDADO}')
    WHERE entidad_id IS NOT NULL AND proveedor_id IS NOT NULL
    LIMIT 10
""").fetchdf()
print(muestra_ruc_limpio.to_string(index=False))

In [ ]:
# --- Agregar columnas de RUC limpio al consolidado (versión final del notebook 1) ---
RUTA_FINAL = os.path.join(CARPETA_PARQUET, "sercop_consolidado_v1.parquet").replace("\\", "/")

con.execute(f"""
    COPY (
        SELECT
            *,
            regexp_extract(entidad_id, 'EC-RUC-(\\d{{13}})', 1)   AS entidad_ruc,
            regexp_extract(proveedor_id, 'EC-RUC-(\\d{{13}})', 1) AS proveedor_ruc
        FROM read_parquet('{RUTA_CONSOLIDADO}')
    )
    TO '{RUTA_FINAL}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# --- Verificación final ---
n_final = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_FINAL}')").fetchone()[0]
assert n_final == total_consolidado, "Descuadre al agregar columnas de RUC -- revisar."
print(f"Archivo final: {RUTA_FINAL}")
print(f"Filas: {n_final:,}  (coincide con el consolidado original)")

cols_final = con.execute(f"SELECT * FROM read_parquet('{RUTA_FINAL}') LIMIT 0").df().columns.tolist()
print(f"\nColumnas finales ({len(cols_final)}): {cols_final}")

pct_ruc = con.execute(f"""
    SELECT
        ROUND(100.0 * SUM(CASE WHEN entidad_ruc IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_entidad_ruc_nulo,
        ROUND(100.0 * SUM(CASE WHEN proveedor_ruc IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_proveedor_ruc_nulo,
        COUNT(DISTINCT entidad_ruc) AS n_entidades_unicas,
        COUNT(DISTINCT proveedor_ruc) AS n_proveedores_unicos
    FROM read_parquet('{RUTA_FINAL}')
""").fetchdf()
print("\nCobertura y cardinalidad de los RUC extraídos:")
print(pct_ruc.to_string(index=False))

In [ ]:
RUTA_FINAL = os.path.join(CARPETA_PARQUET, "sercop_consolidado_v1.parquet").replace("\\", "/")

# --- Diagnóstico: ¿cómo se distinguen las "convocatorias de calificación de proveedores"
#     (registro en convenio marco) de una compra real dentro de catálogo electrónico? ---

# 1. ¿Existen procesos cuyo tipo_contratacion mencione "calificación" o "selección de proveedores"
#    como el PROPÓSITO del proceso (no como el nombre del convenio ya adjudicado)?
candidatos_calificacion = con.execute(f"""
    SELECT tipo_contratacion, metodo_contratacion, COUNT(*) AS n,
           SUM(CASE WHEN monto_tender IS NULL AND monto_award IS NULL THEN 1 ELSE 0 END) AS n_sin_monto,
           SUM(CASE WHEN proveedor_id IS NULL THEN 1 ELSE 0 END) AS n_sin_proveedor
    FROM read_parquet('{RUTA_FINAL}')
    WHERE tipo_contratacion ILIKE '%calificaci%' OR tipo_contratacion ILIKE '%selecci%n de proveedor%'
    GROUP BY 1, 2
    ORDER BY n DESC
    LIMIT 30
""").fetchdf()
print(f"Filas encontradas con patrón de calificación/selección de proveedores: {candidatos_calificacion['n'].sum():,}")
print(candidatos_calificacion.to_string(index=False))

In [ ]:
# 2. Distribución de metodo_contratacion y tender_status para esos candidatos,
#    para ver si se pueden distinguir de una compra real por el status o por el monto
detalle_status = con.execute(f"""
    SELECT metodo_contratacion, tender_status, COUNT(*) AS n,
           ROUND(AVG(CASE WHEN monto_tender IS NULL AND monto_award IS NULL THEN 1.0 ELSE 0.0 END)*100, 1) AS pct_sin_monto
    FROM read_parquet('{RUTA_FINAL}')
    WHERE tipo_contratacion ILIKE '%calificaci%' OR tipo_contratacion ILIKE '%selecci%n de proveedor%'
    GROUP BY 1, 2
    ORDER BY n DESC
""").fetchdf()
print(detalle_status.to_string(index=False))

In [ ]:
# --- ¿Qué valores toma 'etapas' (tag) en todo el consolidado? ---
distintas_etapas = con.execute(f"""
    SELECT etapas, COUNT(*) AS n
    FROM read_parquet('{RUTA_FINAL}')
    GROUP BY 1
    ORDER BY n DESC
    LIMIT 20
""").fetchdf()
print(distintas_etapas.to_string(index=False))

In [ ]:
# --- Procesos que se quedan SOLO en 'planning' (nunca llegan a tender/award/contract) ---
solo_planning = con.execute(f"""
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN monto_tender IS NULL AND monto_award IS NULL THEN 1 ELSE 0 END) AS n_sin_monto,
        SUM(CASE WHEN proveedor_id IS NULL THEN 1 ELSE 0 END) AS n_sin_proveedor
    FROM read_parquet('{RUTA_FINAL}')
    WHERE list_contains(etapas, 'planning')
      AND NOT list_contains(etapas, 'tender')
      AND NOT list_contains(etapas, 'award')
      AND NOT list_contains(etapas, 'contract')
""").fetchdf()
print("Procesos que se quedan solo en 'planning':")
print(solo_planning.to_string(index=False))

# Muestra de tipo_contratacion para esos casos
muestra_planning = con.execute(f"""
    SELECT tipo_contratacion, metodo_contratacion, COUNT(*) AS n
    FROM read_parquet('{RUTA_FINAL}')
    WHERE list_contains(etapas, 'planning')
      AND NOT list_contains(etapas, 'tender')
      AND NOT list_contains(etapas, 'award')
      AND NOT list_contains(etapas, 'contract')
    GROUP BY 1, 2
    ORDER BY n DESC
    LIMIT 20
""").fetchdf()
print("\ntipo_contratacion para procesos solo-planning:")
print(muestra_planning.to_string(index=False))

In [ ]:
CARPETA_RESULTADOS = r"C:\SERCOP_FINAL\03_resultados"
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)

RUTA_SOLO_PLANNING = os.path.join(CARPETA_RESULTADOS, "procesos_solo_planeacion.parquet").replace("\\", "/")
RUTA_BASE_MODELADO = os.path.join(CARPETA_RESULTADOS, "base_modelado_v1.parquet").replace("\\", "/")

# --- Procesos que NO son compras (se quedan solo en planning) ---
con.execute(f"""
    COPY (
        SELECT * FROM read_parquet('{RUTA_FINAL}')
        WHERE list_contains(etapas, 'planning')
          AND NOT list_contains(etapas, 'tender')
          AND NOT list_contains(etapas, 'award')
          AND NOT list_contains(etapas, 'contract')
    )
    TO '{RUTA_SOLO_PLANNING}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# --- Base de modelado: todo lo demás (procesos que sí avanzaron más allá de planning) ---
con.execute(f"""
    COPY (
        SELECT * FROM read_parquet('{RUTA_FINAL}')
        WHERE NOT (
            list_contains(etapas, 'planning')
            AND NOT list_contains(etapas, 'tender')
            AND NOT list_contains(etapas, 'award')
            AND NOT list_contains(etapas, 'contract')
        )
    )
    TO '{RUTA_BASE_MODELADO}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_solo_planning = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_SOLO_PLANNING}')").fetchone()[0]
n_base_modelado = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_BASE_MODELADO}')").fetchone()[0]

print(f"Procesos solo-planning (excluidos):  {n_solo_planning:,}")
print(f"Base de modelado:                    {n_base_modelado:,}")
print(f"Suma:                                {n_solo_planning + n_base_modelado:,}")

assert n_solo_planning + n_base_modelado == n_final, (
    f"Descuadre: {n_solo_planning:,} + {n_base_modelado:,} != {n_final:,}"
)
print("\nOK: la suma de ambos archivos coincide exactamente con el consolidado original.")